https://docs.isaacsim.omniverse.nvidia.com/latest/installation/install_python.html

In [ ]:
# install

%pip install isaacsim[all,extscache]==5.0.0 --extra-index-url https://pypi.nvidia.com

In [ ]:
%pip install isaacsim-replicator

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
import omni.kit.pipapi
omni.kit.pipapi.install("Pillow", module="PIL")
import PIL

In [ ]:
from isaacsim import SimulationApp

# Start the application
simulation_app = SimulationApp({"headless": False})

# Get the utility to enable extensions
from isaacsim.core.utils.extensions import enable_extension

# Enable the layers and stage windows in the UI
enable_extension("omni.kit.usd")
enable_extension("omni.replicator.core")

simulation_app.update()

In [ ]:
import omni.replicator.core as rep

# A simple "Hello World" for Replicator
# This will create a sphere and a cube and place them in the scene.

with rep.new_layer():
    # Define the shapes we want to create
    sphere = rep.create.sphere(position=(0, 0, 100))
    cube = rep.create.cube(position=(150, 0, 100))

    # Use a randomizer to place 10 copies of each shape
    with rep.trigger.on_frame(num_frames=10):
        with rep.distribution.sequence([sphere, cube]):
            rep.modify.pose(
                position=rep.distribution.uniform((-300, 0, -300), (300, 100, 300)),
                scale=rep.distribution.uniform(0.5, 2)
            )

print("Replicator script finished! Check the viewport.")

In [ ]:
import argparse
import json
import os
import sys
from datetime import datetime
from pathlib import Path
from typing import Optional

try:
    import omni.replicator.core as rep
except ModuleNotFoundError as exc:  # pragma: no cover - handled gracefully for docs/tests
    raise SystemExit(
        "omni.replicator.core is not available. "
        "Run this script using an Omniverse Kit python environment."
    ) from exc

In [ ]:
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Generate a YOLO dataset run with Replicator.")
    parser.add_argument(
        "--scene",
        type=Path,
        default=None,
        help="Optional USD scene to load before applying the default synthetic setup.",
    )
    parser.add_argument(
        "--frames",
        type=int,
        default=20,
        help="Number of frames / samples to render.",
    )
    parser.add_argument(
        "--width",
        type=int,
        default=640,
        help="Render product width in pixels.",
    )
    parser.add_argument(
        "--height",
        type=int,
        default=480,
        help="Render product height in pixels.",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=None,
        help="Optional RNG seed for deterministic runs.",
    )
    parser.add_argument(
        "--run-name",
        type=str,
        default=None,
        help="Optional name for this run. Defaults to current timestamp.",
    )
    parser.add_argument(
        "--output-root",
        type=Path,
        default=Path("data/runs"),
        help="Root directory that will contain the generated dataset run.",
    )
    parser.add_argument(
        "--metadata-file",
        type=Path,
        default=None,
        help="Optional JSON file to write run metadata to (in addition to stdout).",
    )
    return parser.parse_args()


def ensure_output_dir(root: Path, run_name: str) -> Path:
    run_dir = root / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir


def build_default_scene():
    """Create a minimal randomized scene if the user did not load a USD."""
    with rep.new_layer():
        dome_light = rep.create.light(
            light_type="Dome",
            rotation=(0, 45, 0),
            intensity=4500,
        )
        plane = rep.create.plane(size=400)
        with rep.get.prims(plane):
            rep.physics.collider()

        cubes = rep.create.cube(count=10)

        with rep.trigger.on_frame():
            with cubes:
                rep.modify.pose(
                    position=rep.distribution.uniform((-150, 10, -150), (150, 160, 150)),
                    rotation=rep.distribution.uniform((0, 0, 0), (360, 360, 360)),
                    scale=rep.distribution.uniform(15, 40),
                )

        with dome_light:
            rep.modify.pose(
                rotation=rep.distribution.uniform((0, 0, 0), (0, 360, 0)),
            )


def configure_scene(scene_path: Optional[Path]):
    if scene_path:
        rep.create.from_usd(str(scene_path))
    else:
        build_default_scene()


def run_dataset(args: argparse.Namespace) -> dict:
    if args.frames <= 0:
        raise ValueError("--frames must be greater than zero")

    if args.seed is not None:
        rep.set_seed(args.seed)

    run_name = args.run_name or datetime.utcnow().strftime("%Y%m%d-%H%M%S")
    output_root = args.output_root.resolve()
    run_dir = ensure_output_dir(output_root, run_name)

    configure_scene(args.scene)

    camera = rep.create.camera(
        position=(0, 300, 400),
        look_at=(0, 0, 0),
        focal_length=35,
    )
    render_product = rep.create.render_product(camera, resolution=(args.width, args.height))

    writer = rep.WriterRegistry.get("Yolo")
    writer.initialize(
        output_dir=str(run_dir),
        colorize_instance_mask=True,
        write_groundtruth_to_json=True,
    )
    writer.attach([render_product])

    rep.orchestrator.run(args.frames)

    images_dir = run_dir / "rgb"
    labels_dir = run_dir / "labels"
    metadata = {
        "run_name": run_name,
        "output_dir": str(run_dir),
        "frames": args.frames,
        "resolution": {"width": args.width, "height": args.height},
        "scene": str(args.scene) if args.scene else "generated_default",
        "images_dir": str(images_dir),
        "labels_dir": str(labels_dir),
    }

    if args.metadata_file:
        args.metadata_file.parent.mkdir(parents=True, exist_ok=True)
        args.metadata_file.write_text(json.dumps(metadata, indent=2))

    return metadata


def main():
    args = parse_args()

    try:
        metadata = run_dataset(args)
    except Exception as exc:  # pragma: no cover - helpful for CLI use
        print(json.dumps({"error": str(exc)}), file=sys.stderr)
        raise

    print(json.dumps(metadata))


if __name__ == "__main__":
    main()


### Merge folders with files the same names

In [ ]:
import os
import shutil
import xml.etree.ElementTree as ET

source = "C:\\Users\\info\\Projects\\replicant\\data\\objects7"
target = "C:\\Users\\info\\Projects\\replicant\\data\\objects"
label = "_7"  # the text to add to filenames


os.makedirs(target, exist_ok=True)

for filename in os.listdir(source):
    src_path = os.path.join(source, filename)
    if not os.path.isfile(src_path):
        continue

    name, ext = os.path.splitext(filename)
    new_name = f"{name}{label}{ext}"
    dst_path = os.path.join(target, new_name)

    if ext.lower() == ".xml":
        try:
            tree = ET.parse(src_path)
            root = tree.getroot()

            # Update <filename> to include the label on the image name
            fn_node = root.find("filename") or root.find(".//filename")
            if fn_node is not None and (fn_node.text or "").strip():
                img_name = fn_node.text.strip()
                img_base, img_ext = os.path.splitext(img_name)
                fn_node.text = f"{img_base}{label}{img_ext}"

            # Optional: update <path> if present
            path_node = root.find("path") or root.find(".//path")
            if path_node is not None and fn_node is not None and fn_node.text:
                path_node.text = os.path.join(target, fn_node.text)

            # Write updated XML to destination with the new XML filename
            tree.write(dst_path, encoding="utf-8", xml_declaration=True)
        except ET.ParseError as e:
            # If parsing fails, just copy the file with the new name
            shutil.copy2(src_path, dst_path)
            print(f"Warning: couldn't parse {filename}: {e}")
    else:
        shutil.copy2(src_path, dst_path)

print("✅ Files copied and XML filenames updated!")

In [ ]:
from skimage import io
from brisque import BRISQUE

score = BRISQUE().score(io.imread("image.jpg"))
print("BRISQUE score:", score)


In [ ]:
from pathlib import Path
from skimage import io
from brisque import BRISQUE

folder = Path(r"../data/objects")
patterns = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff")

files = []
for pat in patterns:
    files.extend(folder.rglob(pat))

if not files:
    print(f"No images found in: {folder.resolve()}")
else:
    b = BRISQUE()
    results = []
    for f in sorted(files):
        try:
            img = io.imread(f)
            score = b.score(img)
            results.append((f, score))
            print(f"{f} -> BRISQUE: {score:.4f}")
        except Exception as e:
            print(f"Failed on {f}: {e}")

    if results:
        best = sorted(results, key=lambda x: x[1])[:5]
        worst = sorted(results, key=lambda x: x[1], reverse=True)[:5]
        print("\nTop 5 (best quality, lowest scores):")
        for f, s in best:
            print(f"- {f} -> {s:.4f}")
        print("\nTop 5 (worst quality, highest scores):")
        for f, s in worst:
            print(f"- {f} -> {s:.4f}")

        # Quality categorization summary
        def brisque_label(s: float) -> str:
            if s < 20:
                return "excellent"
            if s < 40:
                return "good"
            if s < 60:
                return "poor"
            return "bad"

        buckets = {"excellent": 0, "good": 0, "poor": 0, "bad": 0}
        for _, s in results:
            buckets[brisque_label(s)] += 1

        print("\nQuality distribution (BRISQUE):")
        print(f"  0-20:   excellent -> {buckets['excellent']}")
        print(f"  20-40:  good      -> {buckets['good']}")
        print(f"  40-60:  poor      -> {buckets['poor']}")
        print(f"  60-100+: bad      -> {buckets['bad']}")


In [ ]:
import csv

csv_path = folder / "brisque_scores.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    writer.writerow(["path", "brisque_score"])
    for f, score in results:
        writer.writerow([str(f), score])

print(f"Saved CSV to: {csv_path.resolve()}")